# Monte Carlo Methods — Learning from Complete Episodes Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: rollout → list of (s, a, r)

In [ ]:
```python

def rollout(env, policy, max_steps=200):

    trajectory = []

    s = env.reset()

    for _ in range(max_steps):

        a = policy(s)

        s_next, r, done = env.step(s, a)

        trajectory.append((s, a, r))

        s = s_next

        if done:

            break

    return trajectory

In [ ]:
```

No model, only `env.reset()` and `env.step(s, a)`. Same interface as a gym environment but stripped down.

### Step 2: compute returns (reverse sweep)

In [ ]:
```python

def returns_from(trajectory, gamma):

    returns = []

    G = 0.0

    for _, _, r in reversed(trajectory):

        G = r + gamma * G

        returns.append(G)

    return list(reversed(returns))

In [ ]:
```

One pass, `O(T)`. The backward recurrence `G_t = r_{t+1} + γ G_{t+1}` avoids re-summing.

### Step 3: first-visit MC evaluation

In [ ]:
```python

def mc_policy_evaluation(env, policy, episodes, gamma=0.99):

    V = defaultdict(float)

    counts = defaultdict(int)

    for _ in range(episodes):

        trajectory = rollout(env, policy)

        returns = returns_from(trajectory, gamma)

        seen = set()

        for t, ((s, _, _), G) in enumerate(zip(trajectory, returns)):

            if s in seen:

                continue

            seen.add(s)

            counts[s] += 1

            V[s] += (G - V[s]) / counts[s]

    return V

In [ ]:
```

Three lines do the work: mark state as seen on first visit, increment count, update running mean.

### Step 4: ε-greedy MC control (on-policy)

In [ ]:
```python

def mc_control(env, episodes, gamma=0.99, epsilon=0.1):

    Q = defaultdict(lambda: {a: 0.0 for a in ACTIONS})

    counts = defaultdict(lambda: {a: 0 for a in ACTIONS})

    def policy(s):

        if random() < epsilon:

            return choice(ACTIONS)

        return max(Q[s], key=Q[s].get)

    for _ in range(episodes):

        trajectory = rollout(env, policy)

        returns = returns_from(trajectory, gamma)

        seen = set()

        for (s, a, _), G in zip(trajectory, returns):

            if (s, a) in seen:

                continue

            seen.add((s, a))

            counts[s][a] += 1

            Q[s][a] += (G - Q[s][a]) / counts[s][a]

    return Q, policy

In [ ]:
```

### Step 5: compare to DP gold standard

Your MC estimate of `V^π` should agree with the DP result from Lesson 02 as episodes → ∞. In practice: 50,000 episodes on 4×4 GridWorld gets you within `~0.1` of the DP answer.

## Exercises

In [ ]:
1. **Easy.** Implement first-visit MC evaluation of the uniform-random policy on 4×4 GridWorld. Run 10,000 episodes. Plot `V(0,0)` as a function of episode count against the DP answer.
2. **Medium.** Implement ε-greedy MC control with `ε ∈ {0.01, 0.1, 0.3}`. Compare mean return after 20,000 episodes. What does the curve look like? Where does the bias-variance tradeoff live?
3. **Hard.** Implement *off-policy* MC with importance sampling: collect data under uniform-random policy `μ`, estimate `V^π` for the deterministic optimal policy `π`. Compare plain IS vs per-decision IS vs weighted IS. Which has lowest variance?